![](image.jpg)


Dive into the heart of data science with a project that combines healthcare insights and predictive analytics. As a Data Scientist at a top Health Insurance company, you have the opportunity to predict customer healthcare costs using the power of machine learning. Your insights will help tailor services and guide customers in planning their healthcare expenses more effectively.

## Dataset Summary

Meet your primary tool: the `insurance.csv` dataset. Packed with information on health insurance customers, this dataset is your key to unlocking patterns in healthcare costs. Here's what you need to know about the data you'll be working with:

## insurance.csv
| Column    | Data Type | Description                                                      |
|-----------|-----------|------------------------------------------------------------------|
| `age`       | int       | Age of the primary beneficiary.                                  |
| `sex`       | object    | Gender of the insurance contractor (male or female).             |
| `bmi`       | float     | Body mass index, a key indicator of body fat based on height and weight. |
| `children`  | int       | Number of dependents covered by the insurance plan.              |
| `smoker`    | object    | Indicates whether the beneficiary smokes (yes or no).            |
| `region`    | object    | The beneficiary's residential area in the US, divided into four regions. |
| `charges`   | float     | Individual medical costs billed by health insurance.             |



A bit of data cleaning is key to ensure the dataset is ready for modeling. Once your model is built using the `insurance.csv` dataset, the next step is to apply it to the `validation_dataset.csv`. This new dataset, similar to your training data minus the `charges` column, tests your model's accuracy and real-world utility by predicting costs for new customers.

## Let's Get Started!

This project is your playground for applying data science in a meaningful way, offering insights that have real-world applications. Ready to explore the data and uncover insights that could revolutionize healthcare planning? Let's begin this exciting journey!

In [2]:
# Re-run this cell
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Loading the insurance dataset
insurance_data_path = '../data/insurance.csv'
insurance = pd.read_csv(insurance_data_path)
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.924
1,18.0,male,33.770,1.0,no,Southeast,1725.5523
2,28.0,male,33.000,3.0,no,southeast,$4449.462
3,33.0,male,22.705,0.0,no,northwest,$21984.47061
4,32.0,male,28.880,0.0,no,northwest,$3866.8552


In [3]:
#Check the data set info

insurance.info


<bound method DataFrame.info of        age     sex     bmi  children smoker     region       charges
0     19.0  female  27.900       0.0    yes  southwest     16884.924
1     18.0    male  33.770       1.0     no  Southeast     1725.5523
2     28.0    male  33.000       3.0     no  southeast     $4449.462
3     33.0    male  22.705       0.0     no  northwest  $21984.47061
4     32.0    male  28.880       0.0     no  northwest    $3866.8552
...    ...     ...     ...       ...    ...        ...           ...
1333  50.0    male  30.970       3.0     no  Northwest   $10600.5483
1334 -18.0  female  31.920       0.0     no  Northeast     2205.9808
1335  18.0  female  36.850       0.0     no  southeast    $1629.8335
1336  21.0  female  25.800       0.0     no  southwest      2007.945
1337  61.0  female  29.070       0.0    yes  northwest    29141.3603

[1338 rows x 7 columns]>

In [4]:
insurance.describe()

,age,bmi,children
count,1272.000000,1272.000000,1272.000000
mean,35.214623,30.560550,0.948899
std,22.478251,6.095573,1.303532
min,-64.000000,15.960000,-4.000000
25%,24.750000,26.180000,0.000000
50%,38.000000,30.210000,1.000000
75%,51.000000,34.485000,2.000000
max,64.000000,53.130000,5.000000


In [5]:
# Handle missing values

insurance.isnull().sum()

age         66
sex         66
bmi         66
children    66
smoker      66
region      66
charges     54
dtype: int64

In [6]:
# Handle duplicate values

insurance.duplicated().sum()
insurance = insurance.drop_duplicates()


In [7]:
# Check data types of columns

insurance.dtypes

age         float64
sex          object
bmi         float64
children    float64
smoker       object
region       object
charges      object
dtype: object

In [8]:
# check for unique values in categorical columns

insurance.nunique()

age           80
sex            6
bmi          539
children      10
smoker         2
region         8
charges     1272
dtype: int64

In [9]:
insurance['sex'].unique()

array(['female', 'male', 'woman', 'F', 'man', nan, 'M'], dtype=object)

In [10]:
# Defining dictionaries for mapping 

mapping_sex = {
    'woman': 'female',
    'F':'female',
    'man':'male',
    'M':'male',
    'female':'female',
    'male':'male'
}
insurance.loc[:, 'sex'] = insurance['sex'].map(mapping_sex)
insurance['sex'].unique()

array(['female', 'male', nan], dtype=object)

In [11]:
insurance.nunique()


age           80
sex            2
bmi          539
children      10
smoker         2
region         8
charges     1272
dtype: int64

In [12]:
insurance['region'].unique()

array(['southwest', 'Southeast', 'southeast', 'northwest', 'Northwest',
       'Northeast', 'northeast', 'Southwest', nan], dtype=object)

In [13]:
mapping_region ={
    'Southwest':'southwest',
    'Northwest':'northwest',
    'Southeast':'southeast',
    'Northeast':'northeast',
    'southwest':'southwest',
    'northwest':'northwest',
    'southeast':'southeast',
    'northeast':'northeast'

    }
insurance.loc[:,'region']= insurance['region'].map(mapping_region)
insurance['region'].unique()

array(['southwest', 'southeast', 'northwest', 'northeast', nan],
      dtype=object)

In [14]:
insurance

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.924
1,18.0,male,33.770,1.0,no,southeast,1725.5523
2,28.0,male,33.000,3.0,no,southeast,$4449.462
3,33.0,male,22.705,0.0,no,northwest,$21984.47061
4,32.0,male,28.880,0.0,no,northwest,$3866.8552
...,...,...,...,...,...,...,...
1333,50.0,male,30.970,3.0,no,northwest,$10600.5483
1334,-18.0,female,31.920,0.0,no,northeast,2205.9808
1335,18.0,female,36.850,0.0,no,southeast,$1629.8335
1336,21.0,female,25.800,0.0,no,southwest,2007.945


In [15]:
insurance = insurance[insurance['age'] > 0]
insurance = insurance[insurance['children'] > 0]
insurance.describe()

,age,bmi,children
count,629.000000,620.000000,629.000000
mean,39.995231,30.596476,1.904610
std,11.984397,6.140738,0.979305
min,18.000000,16.815000,1.000000
25%,31.000000,26.315000,1.000000
50%,40.000000,30.205000,2.000000
75%,49.000000,34.407500,3.000000
max,64.000000,50.380000,5.000000


In [16]:
insurance['charges']= insurance['charges'].str.replace(r'[$]','', regex=True).astype(float)
insurance

,age,sex,bmi,children,smoker,region,charges
1,18.0,male,33.770,1.0,no,southeast,1725.55230
2,28.0,male,33.000,3.0,no,southeast,4449.46200
6,46.0,female,33.440,1.0,no,southeast,8240.58960
7,37.0,female,27.740,3.0,no,northwest,7281.50560
8,37.0,male,29.830,2.0,no,northeast,6406.41070
...,...,...,...,...,...,...,...
1328,23.0,female,24.225,2.0,no,northeast,22395.74424
1329,52.0,male,38.600,2.0,no,southwest,10325.20600
1330,57.0,female,25.740,2.0,no,southeast,12629.16560
1332,52.0,female,44.700,3.0,no,southwest,11411.68500


In [17]:
insurance['age']= insurance['age'].astype(int)
insurance['children']= insurance['children'].astype(int)
insurance.info()


<class 'pandas.core.frame.DataFrame'>
Index: 629 entries, 1 to 1333
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       629 non-null    int64  
 1   sex       619 non-null    object 
 2   bmi       620 non-null    float64
 3   children  629 non-null    int64  
 4   smoker    620 non-null    object 
 5   region    618 non-null    object 
 6   charges   619 non-null    float64
dtypes: float64(2), int64(2), object(3)
memory usage: 39.3+ KB


In [18]:
insurance.isnull().sum()

age          0
sex         10
bmi          9
children     0
smoker       9
region      11
charges     10
dtype: int64

In [19]:
#Categorical columns should be replaced with mode of the column

insurance['sex']= insurance['sex'].fillna(insurance['sex'].mode()[0])
insurance['smoker']= insurance['smoker'].fillna(insurance['smoker'].mode()[0])
insurance['region']= insurance['region'].fillna(insurance['region'].mode()[0])
insurance.isnull().sum()

age          0
sex          0
bmi          9
children     0
smoker       0
region       0
charges     10
dtype: int64

In [20]:
# Numeric columns should be replace dwith mean of the column

insurance['bmi']= insurance['bmi'].fillna(insurance['bmi'].mean())
insurance['charges']= insurance['charges'].fillna(insurance['charges'].mean())
insurance.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [21]:
insurance.head(20)

,age,sex,bmi,children,smoker,region,charges
1,18,male,33.770000,1,no,southeast,1725.55230
2,28,male,33.000000,3,no,southeast,4449.46200
6,46,female,33.440000,1,no,southeast,8240.58960
7,37,female,27.740000,3,no,northwest,7281.50560
8,37,male,29.830000,2,no,northeast,6406.41070
15,19,male,24.600000,1,no,southwest,1837.23700
16,52,female,30.780000,1,no,northeast,10797.33620
21,30,female,32.400000,1,no,southwest,4149.73600
24,37,male,28.025000,2,no,northwest,6203.90175
25,59,female,27.720000,3,no,southeast,14001.13380


In [22]:
import numpy as np

insurance['charges']= np.log1p(insurance['charges'])

In [23]:
insurance.head()

,age,sex,bmi,children,smoker,region,charges
1,18,male,33.77,1,no,southeast,7.453882
2,28,male,33.00,3,no,southeast,8.400763
6,46,female,33.44,1,no,southeast,9.016949
7,37,female,27.74,3,no,northwest,8.893230
8,37,male,29.83,2,no,northeast,8.765211


In [24]:
# Get dummies for categorical columns
insurance = pd.get_dummies(insurance, columns=['region'], drop_first=True)
map_sex={
    'male':1,
    'female':0
}
map_smoke={
    'yes':1,
    'no':0
}
insurance['sex']= insurance['sex'].map(map_sex)
insurance['smoker']= insurance['smoker'].map(map_smoke)

In [25]:
insurance.head(30)

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
1,18,1,33.770000,1,0,7.453882,False,True,False
2,28,1,33.000000,3,0,8.400763,False,True,False
6,46,0,33.440000,1,0,9.016949,False,True,False
7,37,0,27.740000,3,0,8.893230,True,False,False
8,37,1,29.830000,2,0,8.765211,False,False,False
15,19,1,24.600000,1,0,7.516562,False,False,True
16,52,0,30.780000,1,0,9.287147,False,False,False
21,30,0,32.400000,1,0,8.331041,False,False,True
24,37,1,28.025000,2,0,8.733095,True,False,False
25,59,0,27.720000,3,0,9.546965,False,True,False


In [26]:
insurance['real_charges']= np.expm1(insurance['charges'])
insurance

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest,real_charges
1,18,1,33.770,1,0,7.453882,False,True,False,1725.55230
2,28,1,33.000,3,0,8.400763,False,True,False,4449.46200
6,46,0,33.440,1,0,9.016949,False,True,False,8240.58960
7,37,0,27.740,3,0,8.893230,True,False,False,7281.50560
8,37,1,29.830,2,0,8.765211,False,False,False,6406.41070
...,...,...,...,...,...,...,...,...,...,...
1328,23,0,24.225,2,0,10.016671,False,False,False,22395.74424
1329,52,1,38.600,2,0,9.242440,False,False,True,10325.20600
1330,57,0,25.740,2,0,9.443843,False,True,False,12629.16560
1332,52,0,44.700,3,0,9.342481,False,False,True,11411.68500


In [27]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
num_cols=['age','bmi','children']
insurance[num_cols]=scaler.fit_transform(insurance[num_cols])
insurance[num_cols]

,age,bmi,children
1,-1.836783,0.520956,-0.924462
2,-1.001701,0.394555,1.119427
6,0.501448,0.466784,-0.924462
7,-0.250126,-0.468911,1.119427
8,-0.250126,-0.125822,0.097483
...,...,...,...
1328,-1.419242,-1.045922,0.097483
1329,1.002497,1.313835,0.097483
1330,1.420038,-0.797225,0.097483
1332,1.002497,2.315192,1.119427


In [29]:
insurance.to_csv('../data/processed/insurance_processed.csv', index=False)